In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = r"C:\Users\lola\github\p04_golez_jackwerth_2024\_data"

# -----------------------------
# 1. Load data
# -----------------------------
# Market monthly returns
crsp_msix = pd.read_parquet(f"{DATA_DIR}/CRSP_MSIX.parquet")
crsp_msix["logret_market"] = np.log1p(crsp_msix["vwretd"])
crsp_msix = crsp_msix.set_index("caldt")

# Dividend strip monthly returns
strip = pd.read_parquet(f"{DATA_DIR}/optionm_spx_options.parquet")
strip["month"] = strip["date"].dt.to_period("M")
# use month mid_price 计算收益
strip_monthly = strip.groupby("month")["mid_price"].last().pct_change()
strip_log = np.log1p(strip_monthly)

# Treasury rates
rf = pd.read_parquet(f"{DATA_DIR}/fred_treasury_rates.parquet")
rf["date"] = pd.to_datetime(rf["date"])
rf = rf.set_index("date")

rf_1m = rf["rf_1m"].resample("M").last() / 100        # 1-month risk-free
rf_2y = rf["treasury_2y"].resample("M").last() / 100  # Strip benchmark
rf_10y = rf["treasury_10y"].resample("M").last() / 100 # Market benchmark


# -----------------------------
# 2. Align dates
# -----------------------------
start = max(crsp_msix.index.min(), strip_log.index.min().to_timestamp())
end   = min(crsp_msix.index.max(), strip_log.index.max().to_timestamp())

mkt_log = crsp_msix["logret_market"].loc[start:end]
strip_log = strip_log.loc[start:end]

rf_1m_aligned = rf_1m.loc[start:end]
rf_2y_aligned = rf_2y.loc[start:end]
rf_10y_aligned = rf_10y.loc[start:end]

# -----------------------------
# 3. Excess returns
# -----------------------------
mkt_minus_rf = mkt_log - rf_1m_aligned
strip_minus_rf = strip_log - rf_1m_aligned  # use 1m rf

mkt_minus_treasury = mkt_log - rf_10y_aligned
strip_minus_treasury = strip_log - rf_2y_aligned

# -----------------------------
# 4. Annualized stats
# -----------------------------
def annualize(x):
    return x.mean() * 12

def annualize_std(x):
    return x.std() * np.sqrt(12)

def sharpe_ratio(x):
    return annualize(x) / annualize_std(x)

def ar1(x):
    return x.autocorr(lag=1)

summary = pd.DataFrame({
    "Log return Mean": [annualize(mkt_log)*100, annualize(strip_log)*100],
    "Minus rf Mean": [annualize(mkt_minus_rf)*100, annualize(strip_minus_rf)*100],
    "Minus Treasury Mean": [annualize(mkt_minus_treasury)*100, annualize(strip_minus_treasury)*100],
    
    "Log return Std": [annualize_std(mkt_log)*100, annualize_std(strip_log)*100],
    "Minus rf Std": [annualize_std(mkt_minus_rf)*100, annualize_std(strip_minus_rf)*100],
    "Minus Treasury Std": [annualize_std(mkt_minus_treasury)*100, annualize_std(strip_minus_treasury)*100],
    
    "Log return Sharpe": [sharpe_ratio(mkt_log), sharpe_ratio(strip_log)],
    "Minus rf Sharpe": [sharpe_ratio(mkt_minus_rf), sharpe_ratio(strip_minus_rf)],
    "Minus Treasury Sharpe": [sharpe_ratio(mkt_minus_treasury), sharpe_ratio(strip_minus_treasury)],
    
    "Log return AR1": [ar1(mkt_log), ar1(strip_log)],
    "Minus rf AR1": [ar1(mkt_minus_rf), ar1(strip_minus_rf)],
    "Minus Treasury AR1": [ar1(mkt_minus_treasury), ar1(strip_minus_treasury)],
    
    "N": [len(mkt_log), len(strip_log)]
}, index=["Market","Strip"])

# -----------------------------
# 5. Show table
# -----------------------------
pd.options.display.float_format = '{:.2f}'.format
print(summary)


C:\Users\lola\AppData\Local\Temp\ipykernel_10392\2177038128.py:27: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_1m = rf["rf_1m"].resample("M").last() / 100        # 1-month risk-free
C:\Users\lola\AppData\Local\Temp\ipykernel_10392\2177038128.py:28: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_2y = rf["treasury_2y"].resample("M").last() / 100  # Strip benchmark
C:\Users\lola\AppData\Local\Temp\ipykernel_10392\2177038128.py:29: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_10y = rf["treasury_10y"].resample("M").last() / 100 # Market benchmark


        Log return Mean  Minus rf Mean  Minus Treasury Mean  Log return Std  \
Market             9.41           8.50                 9.59           16.14   
Strip             12.97            NaN                  NaN           69.21   

        Minus rf Std  Minus Treasury Std  Log return Sharpe  Minus rf Sharpe  \
Market         16.13               16.56               0.58             0.53   
Strip            NaN                 NaN               0.19              NaN   

        Minus Treasury Sharpe  Log return AR1  Minus rf AR1  \
Market                   0.58            0.04          0.01   
Strip                     NaN           -0.25           NaN   

        Minus Treasury AR1    N  
Market               -0.02  347  
Strip                  NaN  348  


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = r"C:\Users\lola\github\p04_golez_jackwerth_2024\_data"

# -----------------------------
# 1. Load data
# -----------------------------
# Market monthly returns
crsp_msix = pd.read_parquet(f"{DATA_DIR}/CRSP_MSIX.parquet")
crsp_msix["logret_market"] = np.log1p(crsp_msix["vwretd"])
crsp_msix = crsp_msix.set_index("caldt")

# Dividend strip monthly returns
strip = pd.read_parquet(f"{DATA_DIR}/optionm_spx_options.parquet")
strip["month"] = strip["date"].dt.to_period("M")
# use month mid_price 计算收益
strip_monthly = strip.groupby("month")["mid_price"].last().pct_change()
strip_log = np.log1p(strip_monthly)

# Treasury rates
rf = pd.read_parquet(f"{DATA_DIR}/fred_treasury_rates.parquet")
rf["date"] = pd.to_datetime(rf["date"])
rf = rf.set_index("date")

rf_1m = rf["rf_1m"].resample("M").last() / 100        # 1-month risk-free
rf_2y = rf["treasury_2y"].resample("M").last() / 100  # Strip benchmark
rf_10y = rf["treasury_10y"].resample("M").last() / 100 # Market benchmark


# -----------------------------
# 2. Align dates
# -----------------------------
start = pd.Period("1996-01", freq="M").to_timestamp()
# Manually set end date to December 2022
end = pd.Period("2022-12", freq="M").to_timestamp()

mkt_log = crsp_msix["logret_market"].loc[start:end]
strip_log = strip_log.loc[start:end]

rf_1m_aligned = rf_1m.loc[start:end]
rf_2y_aligned = rf_2y.loc[start:end]
rf_10y_aligned = rf_10y.loc[start:end]

# -----------------------------
# 3. Excess returns
# -----------------------------
mkt_minus_rf = mkt_log - rf_1m_aligned
strip_minus_rf = strip_log - rf_1m_aligned  # use 1m rf

mkt_minus_treasury = mkt_log - rf_10y_aligned
strip_minus_treasury = strip_log - rf_2y_aligned

# -----------------------------
# 4. Annualized stats
# -----------------------------
def annualize(x):
    return x.mean() * 12

def annualize_std(x):
    return x.std() * np.sqrt(12)

def sharpe_ratio(x):
    return annualize(x) / annualize_std(x)

def ar1(x):
    return x.autocorr(lag=1)

summary = pd.DataFrame({
    "Log return Mean": [annualize(mkt_log)*100, annualize(strip_log)*100],
    "Minus rf Mean": [annualize(mkt_minus_rf)*100, annualize(strip_minus_rf)*100],
    "Minus Treasury Mean": [annualize(mkt_minus_treasury)*100, annualize(strip_minus_treasury)*100],
    
    "Log return Std": [annualize_std(mkt_log)*100, annualize_std(strip_log)*100],
    "Minus rf Std": [annualize_std(mkt_minus_rf)*100, annualize_std(strip_minus_rf)*100],
    "Minus Treasury Std": [annualize_std(mkt_minus_treasury)*100, annualize_std(strip_minus_treasury)*100],
    
    "Log return Sharpe": [sharpe_ratio(mkt_log), sharpe_ratio(strip_log)],
    "Minus rf Sharpe": [sharpe_ratio(mkt_minus_rf), sharpe_ratio(strip_minus_rf)],
    "Minus Treasury Sharpe": [sharpe_ratio(mkt_minus_treasury), sharpe_ratio(strip_minus_treasury)],
    
    "Log return AR1": [ar1(mkt_log), ar1(strip_log)],
    "Minus rf AR1": [ar1(mkt_minus_rf), ar1(strip_minus_rf)],
    "Minus Treasury AR1": [ar1(mkt_minus_treasury), ar1(strip_minus_treasury)],
    
    "N": [len(mkt_log), len(strip_log)]
}, index=["Market","Strip"])

# -----------------------------
# 5. Show table
# -----------------------------
pd.options.display.float_format = '{:.2f}'.format
display(summary)

C:\Users\lola\AppData\Local\Temp\ipykernel_10392\1985935177.py:27: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_1m = rf["rf_1m"].resample("M").last() / 100        # 1-month risk-free
C:\Users\lola\AppData\Local\Temp\ipykernel_10392\1985935177.py:28: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_2y = rf["treasury_2y"].resample("M").last() / 100  # Strip benchmark
C:\Users\lola\AppData\Local\Temp\ipykernel_10392\1985935177.py:29: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_10y = rf["treasury_10y"].resample("M").last() / 100 # Market benchmark


,Log return Mean,Minus rf Mean,Minus Treasury Mean,Log return Std,Minus rf Std,Minus Treasury Std,Log return Sharpe,Minus rf Sharpe,Minus Treasury Sharpe,Log return AR1,Minus rf AR1,Minus Treasury AR1,N
Market,8.62,7.29,8.69,16.29,16.32,16.75,0.53,0.45,0.52,0.05,0.04,0.00,323
Strip,17.42,NaN,NaN,68.87,NaN,NaN,0.25,NaN,NaN,-0.28,NaN,NaN,324


In [5]:
print(strip.columns)


Index(['date', 'exdate', 'cp_flag', 'strike', 'best_bid', 'best_offer',
       'volume', 'open_interest', 'impl_volatility', 'mid_price', 'year',
       'days_to_maturity', 'month'],
      dtype='object')


In [6]:
print(strip.head())

        date     exdate cp_flag  strike  best_bid  best_offer  volume  \
0 1996-01-04 1996-06-22       C  400.00    218.50      219.50    0.00   
1 1996-01-04 1996-06-22       C  425.00    194.38      195.38    0.00   
2 1996-01-04 1996-06-22       C  450.00    170.12      171.12    0.00   
3 1996-01-04 1996-06-22       C  475.00    146.12      147.12    0.00   
4 1996-01-04 1996-06-22       C  500.00    122.25      123.25    0.00   

   open_interest  impl_volatility  mid_price  year  days_to_maturity    month  
0          29.00              NaN     219.00  1996               170  1996-01  
1         360.00              NaN     194.88  1996               170  1996-01  
2        1911.00              NaN     170.62  1996               170  1996-01  
3        2700.00              NaN     146.62  1996               170  1996-01  
4        2990.00              NaN     122.75  1996               170  1996-01  


In [9]:
import pandas as pd
import numpy as np

DATA_DIR = r"C:\Users\lola\github\p04_golez_jackwerth_2024\_data"

# =============================
# 1. LOAD DATA
# =============================

# Market
crsp_msix = pd.read_parquet(f"{DATA_DIR}/CRSP_MSIX.parquet")
crsp_msix["logret_market"] = np.log1p(crsp_msix["vwretd"])
crsp_msix = crsp_msix.set_index("caldt")

# Risk-free & Treasury
rf = pd.read_parquet(f"{DATA_DIR}/fred_treasury_rates.parquet")
rf["date"] = pd.to_datetime(rf["date"])
rf = rf.set_index("date")

rf_1m  = rf["rf_1m"].resample("M").last() / 100
rf_2y  = rf["treasury_2y"].resample("M").last() / 100
rf_10y = rf["treasury_10y"].resample("M").last() / 100

# Options
strip = pd.read_parquet(f"{DATA_DIR}/optionm_spx_options.parquet")
strip["date"] = pd.to_datetime(strip["date"])


# =============================
# Construct 2Y strip properly
# =============================

strip["date"] = pd.to_datetime(strip["date"])
strip["month"] = strip["date"].dt.to_period("M")

# Keep CALL options only
calls = strip[strip["cp_flag"] == "C"].copy()
calls = calls.dropna(subset=["mid_price"])

def compute_strip_month(group):

    # pick maturity closest to 2 years
    target = 730
    group["maturity_diff"] = np.abs(group["days_to_maturity"] - target)
    
    # find exdate with smallest diff
    best_exdate = group.loc[group["maturity_diff"].idxmin(), "exdate"]
    sub = group[group["exdate"] == best_exdate].copy()
    
    sub = sub.sort_values("strike")
    
    K = sub["strike"].values
    C = sub["mid_price"].values
    
    if len(K) < 15:
        return np.nan
    
    integrand = C / (K**2)
    price = np.trapz(integrand, K)
    
    return price

strip_price = calls.groupby("month").apply(compute_strip_month)
strip_price = strip_price.dropna()

strip_log = np.log(strip_price / strip_price.shift(1))
strip_log = strip_log.dropna()
strip_log.index = strip_log.index.to_timestamp()


# =============================
# 3. SAMPLE PERIOD
# =============================

start = pd.Timestamp("1996-01-31")
end   = pd.Timestamp("2022-12-31")

mkt_log   = crsp_msix["logret_market"].loc[start:end]
strip_log = strip_log.loc[start:end]

rf_1m  = rf_1m.loc[start:end]
rf_2y  = rf_2y.loc[start:end]
rf_10y = rf_10y.loc[start:end]


# =============================
# 4. EXCESS RETURNS
# =============================

mkt_minus_rf   = mkt_log - rf_1m
strip_minus_rf = strip_log - rf_1m

mkt_minus_treasury   = mkt_log - rf_10y
strip_minus_treasury = strip_log - rf_2y


# =============================
# 5. SUMMARY STATS
# =============================

def annualize(x):
    return x.mean() * 12

def annualize_std(x):
    return x.std() * np.sqrt(12)

def sharpe_ratio(x):
    return annualize(x) / annualize_std(x)

def ar1(x):
    return x.autocorr(lag=1)

summary = pd.DataFrame({
    
    # Log return
    "Log Mean": [annualize(mkt_log)*100, annualize(strip_log)*100],
    "Log Std":  [annualize_std(mkt_log)*100, annualize_std(strip_log)*100],
    "Log AR1":  [ar1(mkt_log), ar1(strip_log)],
    
    # Minus rf
    "Minus rf Mean": [annualize(mkt_minus_rf)*100, annualize(strip_minus_rf)*100],
    "Minus rf Std":  [annualize_std(mkt_minus_rf)*100, annualize_std(strip_minus_rf)*100],
    "Minus rf Sharpe": [sharpe_ratio(mkt_minus_rf), sharpe_ratio(strip_minus_rf)],
    "Minus rf AR1": [ar1(mkt_minus_rf), ar1(strip_minus_rf)],
    
    # Minus Treasury
    "Minus Tsy Mean": [annualize(mkt_minus_treasury)*100, annualize(strip_minus_treasury)*100],
    "Minus Tsy Std":  [annualize_std(mkt_minus_treasury)*100, annualize_std(strip_minus_treasury)*100],
    "Minus Tsy Sharpe": [sharpe_ratio(mkt_minus_treasury), sharpe_ratio(strip_minus_treasury)],
    "Minus Tsy AR1": [ar1(mkt_minus_treasury), ar1(strip_minus_treasury)],
    
    "N": [len(mkt_log), len(strip_log)]

}, index=["Market","Strip"])

pd.options.display.float_format = '{:.2f}'.format
print(summary)


C:\Users\lola\AppData\Local\Temp\ipykernel_10392\2160874036.py:20: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_1m  = rf["rf_1m"].resample("M").last() / 100
C:\Users\lola\AppData\Local\Temp\ipykernel_10392\2160874036.py:21: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_2y  = rf["treasury_2y"].resample("M").last() / 100
C:\Users\lola\AppData\Local\Temp\ipykernel_10392\2160874036.py:22: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_10y = rf["treasury_10y"].resample("M").last() / 100


        Log Mean  Log Std  Log AR1  Minus rf Mean  Minus rf Std  \
Market      8.37    16.32     0.04           7.29         16.32   
Strip      22.17   236.50    -0.04            NaN           NaN   

        Minus rf Sharpe  Minus rf AR1  Minus Tsy Mean  Minus Tsy Std  \
Market             0.45          0.04            8.69          16.75   
Strip               NaN           NaN             NaN            NaN   

        Minus Tsy Sharpe  Minus Tsy AR1    N  
Market              0.52           0.00  324  
Strip                NaN            NaN  323  


C:\Users\lola\AppData\Local\Temp\ipykernel_10392\2160874036.py:63: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  strip_price = calls.groupby("month").apply(compute_strip_month)
